# Pretraining on two T4 cards with one client process

Train the base model on both T4 cards of a Kaggle session, then publish the
base stage to the Hugging Face Hub.

One process owns both cards. The C++ mesh training driver builds one model
replica per device, one device thread per replica, and reduces the gradients
in process (`docs/client-runtime-design.md`). The driver also owns the step
loop, the metric record, and the summary. Training runs in a child process
through `tools/nanochat client`, so the cells stay responsive.

The balanced two-T4 shape is a 252M-parameter model and 500 steps. A full run
takes a few hours and about 9.1 GiB of each 16 GiB card.

This notebook is one of three. Run `eval.ipynb` to measure the published stage,
and `posttrain.ipynb` to fine-tune it. The container and the Hub layout are in
`docs/checkpoint-format.md`.


## Setup

The setup clones the repository, reads the Kaggle secrets, builds the shared
library for the T4 (`sm_75`), and checks the card count. See
`docs/checkpoint-format.md` section 15.


In [ ]:
# The repository must exist before this module imports. Clone it, add the
# package paths, then let the session module own the host setup and the build.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ming6ao/nanochat.cpp"
REPO = Path("/kaggle/working/nanochat.cpp")
BASE = Path("/kaggle/working/nanochat-cache")

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
sys.path[:0] = [str(REPO / "python"), str(REPO / "notebooks")]

import session

sess = session.bootstrap(REPO_URL, REPO, arch="sm_75", devices=2,
                         base=BASE)
session.describe(sess)

import hub_checkpoint
import nanochat_cpp as nc


## Data and tokenizer

Stage one ClimbMix train shard and one validation shard, then pull the
tokenizer from the Hub. The tokenizer lives in
`<namespace>/nanochat-tokenizer` as one file for each vocabulary size. A pull
miss trains the tokenizer and pushes it. The download is the one fixed cost.


In [ ]:
import os
import shutil

import hub_checkpoint

VOCAB_SIZE = 32768
DATA = BASE / "base_data_climbmix"
TRAIN_SHARD = hub_checkpoint.TRAIN_SHARD
VAL_SHARD = hub_checkpoint.VAL_SHARD

if not (DATA / VAL_SHARD).is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "huggingface_hub"], check=True)
hub_checkpoint.download_climbmix(DATA, token=os.environ.get("HF_TOKEN"))

TOKENIZER = BASE / "tokenizer" / "tokenizer.nctoken"
TOKENIZER.parent.mkdir(parents=True, exist_ok=True)
TOKENIZER_REPO = hub_checkpoint.repo_id_for("tokenizer")

if not TOKENIZER.is_file():
    pulled = hub_checkpoint.pull_tokenizer(
        TOKENIZER_REPO, VOCAB_SIZE, TOKENIZER.parent,
        token=os.environ.get("HF_TOKEN"))
    if pulled is not None:
        shutil.copyfile(pulled, TOKENIZER)
        print("pulled tokenizer:", pulled)
    else:
        print("no tokenizer on the Hub; training one")
        nc.toolchain.build(["//src/tokenizer:tok_train_main"])
        nc.toolchain.run("t0-cpu", [
            "bazel-bin/src/tokenizer/tok_train_main",
            "--parquet", f"{DATA}/{TRAIN_SHARD}",
            "--vocab-size", str(VOCAB_SIZE),
            "--max-chars", "2000000000",
            "--out", str(TOKENIZER),
        ])
        hub_checkpoint.push_tokenizer(TOKENIZER_REPO, VOCAB_SIZE, TOKENIZER,
                                      token=os.environ.get("HF_TOKEN"))

tokenizer = nc.Tokenizer.load(str(TOKENIZER))
assert tokenizer.decode(tokenizer.encode("The capital of France is")) == \
    "The capital of France is", "the tokenizer round trip failed"
print("train:", DATA / TRAIN_SHARD)
print("tokenizer vocab:", tokenizer.vocab_size)


## Model

Change the knobs to try another shape. The default is the balanced two-T4
shape: 16 layers, hidden 1024, 8 heads, 4 key/value heads, and 4096 tokens,
about 252M parameters. Keep `num_heads` and `num_kv_heads` as divisors of
`hidden_dim`.


In [ ]:
import args
import model_fit

MODEL = args.ModelArgs(num_layers=16, hidden_dim=1024, num_heads=8,
                       num_kv_heads=4, seq_len=4096, vocab_size=VOCAB_SIZE)
spec = model_fit.spec_from(MODEL)
config = MODEL.to_config()

counts = model_fit.parameter_counts(spec)
t4 = model_fit.hardware("t4")
print("parameters:", f"{counts['total']:,}")
print("flops per token:", f"{model_fit.flops_per_token(spec) / 1e9:.3f} GFLOP")
print("estimated card memory at per-rank batch 1:",
      f"{model_fit.card_bytes(spec, 1, 'fp32') / 2**30:.2f} GiB of 16 GiB")


## Train

The C++ driver takes the global batch. `per_rank_batch * grad_accum *
world_size` is the global sequence count, split across the two cards. The
driver reduces the gradients once per step.

The cell builds the driver config and starts it in a child process. It
streams the per-step log from the driver. Interrupt the cell to stop the
worker; the driver saves the checkpoint first.


In [ ]:
import mesh_train

RUNS = Path("/kaggle/working/runs")
BASE_STEPS = 500
RUN_NAME = (f"base-L{MODEL.num_layers}-H{MODEL.hidden_dim}"
            f"-n{MODEL.num_heads}-kv{MODEL.num_kv_heads}"
            f"-s{MODEL.seq_len}")

train_config = nc.MeshTrainConfig(
    MODEL.to_config(), preset="track3", world_size=2,
    tokenizer=str(TOKENIZER), train_parquet=f"{DATA}/{TRAIN_SHARD}",
    per_rank_batch=1, grad_accum=8, steps=BASE_STEPS,
    seed=42, data_seed=42, log_every=1, save_every=50,
    resume=True, out_dir=str(RUNS / RUN_NAME),
    flops_per_token=model_fit.flops_per_token(spec),
    peak_flops=t4.peak_flops("fp32"))
print("run:", train_config.out_dir, "| tokens per step:",
      train_config.tokens_per_step)

live = mesh_train.launch(train_config, root=REPO, profile="train")
live.stream()

returncode = live.wait()
print("worker exit code:", returncode)
if returncode != 0:
    print(live.tail(60))

# The per-step log streamed above. This block prints the run summary.
records = live.rows()
summary = live.summary()
print(f"steps {len(records)}  "
      f"mean loss {float(summary.get('mean_loss', 0.0)):.4f}  "
      f"steady MFU {float(summary.get('steady_mfu', 0.0)) * 100:.2f}%")
if summary:
    digests = summary["replica_digests"]
    print("replica digests:", digests)
    assert len(digests) == 2 and len(set(digests)) == 1, (
        "the two replicas differ: the gradient reduce is broken")
checkpoint = Path(summary.get("checkpoint")
                  or train_config.checkpoint_path())
if checkpoint.is_file():
    print("checkpoint:", checkpoint, checkpoint.stat().st_size, "bytes")


## Publish

Write the model arguments, the training job, the run record, the tokenizer, and
the model card beside the checkpoint, then upload the `base` stage and tag it.


In [ ]:
run_dir = Path(train_config.out_dir)
record = hub_checkpoint.write_stage(
    run_dir, stage="base", run_name=RUN_NAME,
    model_json=MODEL.to_json(), train_json=train_config.job_json(),
    tokenizer=TOKENIZER, git_revision=sess.revision,
    build_key=sess.build_key, cuda_arch="sm_75", step=BASE_STEPS,
    created_utc=hub_checkpoint.now_utc(),
    description=f"Base model, pretrained on ClimbMix for {BASE_STEPS} steps.")
print("run directory:", run_dir)
print(record)


In [ ]:
REPO_ID = hub_checkpoint.repo_id_for(RUN_NAME)
hub_checkpoint.push_stage(REPO_ID, "base", run_dir,
                          token=os.environ.get("HF_TOKEN"),
                          tag=f"base-{BASE_STEPS}")
print("pushed:", REPO_ID, "stage base", f"tag base-{BASE_STEPS}")


## Sanity check

Load the checkpoint into one `nc.Model` on card 0, measure a few validation
steps, and print two greedy samples.


In [ ]:
import kaggle_demo

model = nc.Model(config, device="cuda", seed=0)
model.load(str(train_config.checkpoint_path()))
print("device:", model.device, "parameters:", f"{model.param_total():,}")

report = nc.evaluate(model, parquet=str(DATA / VAL_SHARD), tokenizer=tokenizer,
                     tokens=config.seq_len * 4, batch=1, seed=43)
print(f"validation bits per byte: {report.bpb:.4f}")

kaggle_demo.print_samples(model, tokenizer,
                          kaggle_demo.SAMPLE_PROMPTS[:2])


## Next

Run `eval.ipynb` in a later session. It downloads the tag `base-500` and
measures the base stage. Run `posttrain.ipynb` to fine-tune it into the `sft`
stage.
